In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
stations = pd.read_excel("../../data/raw/ev_charging_dataset.xlsx", sheet_name="stations")
sessions = pd.read_excel("../../data/raw/ev_charging_dataset.xlsx", sheet_name="sessions")

In [3]:
print("Stations Dataset:", stations.shape)
print("Sessions Dataset:", sessions.shape)

display(stations.head())
display(sessions.head())

Stations Dataset: (33, 5)
Sessions Dataset: (294024, 15)


,station_id,station_name,region,activation_year,n_chargers
0,STN_001,EV FastCharge 1,North West,2022,4
1,STN_002,EV FastCharge 2,London,2022,5
2,STN_003,EV FastCharge 3,Midlands,2022,5
3,STN_004,EV FastCharge 4,North West,2022,8
4,STN_005,EV FastCharge 5,North West,2022,8


,session_id,station_id,charger_id,charger_type,start_timestamp,end_timestamp,session_duration_minutes,energy_kwh,price_per_kwh,total_cost,payment_method,user_type,customer_id,station_region,year
0,S_00000001,STN_001,STN_001_CH_02,DC_150kW,2022-01-01 08:30:00,2022-01-01 08:56:00,26,56.90,0.47,26.74,app,fleet,CUST_00134,North West,2022
1,S_00000002,STN_001,STN_001_CH_02,DC_150kW,2022-01-01 09:20:00,2022-01-01 09:41:00,21,40.70,0.46,18.72,app,delivery,CUST_00195,North West,2022
2,S_00000003,STN_001,STN_001_CH_04,DC_50kW,2022-01-01 09:00:00,2022-01-01 09:48:00,48,30.90,0.47,14.52,rfid,public,CUST_01935,North West,2022
3,S_00000004,STN_001,STN_001_CH_01,DC_50kW,2022-01-01 23:20:00,2022-01-02 00:01:00,41,28.40,0.45,12.78,rfid,public,CUST_00036,North West,2022
4,S_00000005,STN_001,STN_001_CH_01,DC_50kW,2022-01-01 16:10:00,2022-01-01 17:00:00,50,31.70,0.56,17.75,rfid,taxi,CUST_00011,North West,2022


In [4]:
stations.info()
sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   station_id       33 non-null     object
 1   station_name     33 non-null     object
 2   region           33 non-null     object
 3   activation_year  33 non-null     int64 
 4   n_chargers       33 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 1.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294024 entries, 0 to 294023
Data columns (total 15 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   session_id                294024 non-null  object        
 1   station_id                294024 non-null  object        
 2   charger_id                294024 non-null  object        
 3   charger_type              294024 non-null  object        
 4   start_timestamp           294024 non-nu

In [5]:
print("Missing values:")
display(sessions.isnull().sum())

print("\nEmpty string counts:")
display(
    sessions.select_dtypes(include="object")
    .apply(lambda col: col.astype(str).str.strip().eq("").sum())
)

print("\nDuplicate station IDs:", stations["station_id"].duplicated().sum())
print("Duplicate session IDs:", sessions["session_id"].duplicated().sum())

print("\nUnique stations in sessions:", sessions["station_id"].nunique())
print("Stations in infrastructure dataset:", stations["station_id"].nunique())

print("\nSessions with unknown station IDs:",
      (~sessions["station_id"].isin(stations["station_id"])).sum())

Missing values:


session_id                  0
station_id                  0
charger_id                  0
charger_type                0
start_timestamp             0
end_timestamp               0
session_duration_minutes    0
energy_kwh                  0
price_per_kwh               0
total_cost                  0
payment_method              0
user_type                   0
customer_id                 0
station_region              0
year                        0
dtype: int64


Empty string counts:


session_id        0
station_id        0
charger_id        0
charger_type      0
payment_method    0
user_type         0
customer_id       0
station_region    0
dtype: int64


Duplicate station IDs: 0
Duplicate session IDs: 0

Unique stations in sessions: 33
Stations in infrastructure dataset: 33

Sessions with unknown station IDs: 0


In [6]:
region_check = (
    sessions[["station_id", "station_region"]]
    .drop_duplicates()
    .merge(
        stations[["station_id", "region"]],
        on="station_id",
        how="left"
    )
)

region_check["region_match"] = (
    region_check["station_region"] == region_check["region"]
)

display(region_check)

print("Region mismatches:", (~region_check["region_match"]).sum())

,station_id,station_region,region,region_match
0,STN_001,North West,North West,True
1,STN_002,London,London,True
2,STN_003,Midlands,Midlands,True
3,STN_004,North West,North West,True
4,STN_005,North West,North West,True
5,STN_006,South West,South West,True
6,STN_007,North East,North East,True
7,STN_008,Midlands,Midlands,True
8,STN_009,Wales,Wales,True
9,STN_010,North East,North East,True


Region mismatches: 0


In [7]:
print("Total chargers:", stations["n_chargers"].sum())

print("\nChargers per station:")
display(stations["n_chargers"].describe())

print("\nStations by activation year:")
display(stations["activation_year"].value_counts().sort_index())

print("\nStations by region:")
display(
    stations.groupby("region")
    .agg(
        total_stations=("station_id", "count"),
        total_chargers=("n_chargers", "sum")
    )
    .sort_values("total_stations", ascending=False)
)

Total chargers: 199

Chargers per station:


count   33.00
mean     6.03
std      1.67
min      4.00
25%      4.00
50%      6.00
75%      8.00
max      8.00
Name: n_chargers, dtype: float64


Stations by activation year:


activation_year
2022     8
2023     9
2024    16
Name: count, dtype: int64


Stations by region:


,total_stations,total_chargers
region,,
North West,7,48
London,5,25
Midlands,5,26
South East,4,26
North East,3,17
Scotland,3,22
South West,3,15
Yorkshire & Humber,2,12
Wales,1,8


In [8]:
station_performance = (
    sessions.groupby("station_id")
    .agg(
        total_sessions=("session_id", "count"),
        total_energy_kwh=("energy_kwh", "sum"),
        total_revenue=("total_cost", "sum"),
        total_duration_minutes=("session_duration_minutes", "sum"),
        average_energy_per_session=("energy_kwh", "mean"),
        average_revenue_per_session=("total_cost", "mean")
    )
    .reset_index()
)

station_performance = stations.merge(
    station_performance,
    on="station_id",
    how="left"
)

station_performance["sessions_per_charger"] = (
    station_performance["total_sessions"] /
    station_performance["n_chargers"]
)

station_performance["energy_per_charger"] = (
    station_performance["total_energy_kwh"] /
    station_performance["n_chargers"]
)

station_performance["revenue_per_charger"] = (
    station_performance["total_revenue"] /
    station_performance["n_chargers"]
)

display(station_performance.head())

,station_id,station_name,region,activation_year,n_chargers,total_sessions,total_energy_kwh,total_revenue,total_duration_minutes,average_energy_per_session,average_revenue_per_session,sessions_per_charger,energy_per_charger,revenue_per_charger
0,STN_001,EV FastCharge 1,North West,2022,4,15162,"620,951.60","359,914.16",560630,40.95,23.74,"3,790.50","155,237.90","89,978.54"
1,STN_002,EV FastCharge 2,London,2022,5,15276,"940,999.00","545,458.72",405401,61.60,35.71,"3,055.20","188,199.80","109,091.74"
2,STN_003,EV FastCharge 3,Midlands,2022,5,15321,"733,802.60","424,603.28",513898,47.90,27.71,"3,064.20","146,760.52","84,920.66"
3,STN_004,EV FastCharge 4,North West,2022,8,15121,"720,723.50","417,831.11",540499,47.66,27.63,"1,890.12","90,090.44","52,228.89"
4,STN_005,EV FastCharge 5,North West,2022,8,15160,"702,048.50","406,249.04",503018,46.31,26.80,"1,895.00","87,756.06","50,781.13"


In [9]:
print("Station performance shape:", station_performance.shape)

print("\nTotal sessions:")
print(station_performance["total_sessions"].sum(), sessions.shape[0])

print("\nTotal energy:")
print(station_performance["total_energy_kwh"].sum(), sessions["energy_kwh"].sum())

print("\nTotal revenue:")
print(station_performance["total_revenue"].sum(), sessions["total_cost"].sum())

print("\nMissing values:")
display(station_performance.isnull().sum())

print("\nDuplicate station IDs:",
      station_performance["station_id"].duplicated().sum())

Station performance shape: (33, 14)

Total sessions:
294024 294024

Total energy:
13808745.799999999 13808745.8

Total revenue:
8171100.12 8171100.119999999

Missing values:


station_id                     0
station_name                   0
region                         0
activation_year                0
n_chargers                     0
total_sessions                 0
total_energy_kwh               0
total_revenue                  0
total_duration_minutes         0
average_energy_per_session     0
average_revenue_per_session    0
sessions_per_charger           0
energy_per_charger             0
revenue_per_charger            0
dtype: int64


Duplicate station IDs: 0


In [10]:
performance_metrics = [
    "total_sessions",
    "total_energy_kwh",
    "total_revenue",
    "sessions_per_charger",
    "energy_per_charger",
    "revenue_per_charger"
]

for metric in performance_metrics:
    print(f"\nTop 5 stations by {metric}:")
    display(
        station_performance[
            ["station_id", "station_name", "region", "n_chargers", metric]
        ]
        .sort_values(metric, ascending=False)
        .head(5)
    )

    print(f"Bottom 5 stations by {metric}:")
    display(
        station_performance[
            ["station_id", "station_name", "region", "n_chargers", metric]
        ]
        .sort_values(metric)
        .head(5)
    )


Top 5 stations by total_sessions:


,station_id,station_name,region,n_chargers,total_sessions
2,STN_003,EV FastCharge 3,Midlands,5,15321
1,STN_002,EV FastCharge 2,London,5,15276
6,STN_007,EV FastCharge 7,North East,4,15218
0,STN_001,EV FastCharge 1,North West,4,15162
4,STN_005,EV FastCharge 5,North West,8,15160


Bottom 5 stations by total_sessions:


,station_id,station_name,region,n_chargers,total_sessions
20,STN_021,EV FastCharge 21,South East,8,4927
30,STN_031,EV FastCharge 31,Midlands,4,4999
27,STN_028,EV FastCharge 28,Midlands,4,5000
26,STN_027,EV FastCharge 27,South East,8,5007
24,STN_025,EV FastCharge 25,North West,8,5036



Top 5 stations by total_energy_kwh:


,station_id,station_name,region,n_chargers,total_energy_kwh
1,STN_002,EV FastCharge 2,London,5,"940,999.00"
7,STN_008,EV FastCharge 8,Midlands,5,"854,549.00"
5,STN_006,EV FastCharge 6,South West,4,"778,478.10"
2,STN_003,EV FastCharge 3,Midlands,5,"733,802.60"
3,STN_004,EV FastCharge 4,North West,8,"720,723.50"


... (14 outputs left out)


Top 5 stations by revenue_per_charger:


,station_id,station_name,region,n_chargers,revenue_per_charger
5,STN_006,EV FastCharge 6,South West,4,"112,579.75"
1,STN_002,EV FastCharge 2,London,5,"109,091.74"
7,STN_008,EV FastCharge 8,Midlands,5,"99,063.04"
6,STN_007,EV FastCharge 7,North East,4,"96,485.80"
0,STN_001,EV FastCharge 1,North West,4,"89,978.54"


Bottom 5 stations by revenue_per_charger:


,station_id,station_name,region,n_chargers,revenue_per_charger
20,STN_021,EV FastCharge 21,South East,8,"14,054.50"
22,STN_023,EV FastCharge 23,Scotland,8,"15,625.51"
25,STN_026,EV FastCharge 26,London,8,"16,002.70"
23,STN_024,EV FastCharge 24,North West,7,"18,184.60"
24,STN_025,EV FastCharge 25,North West,8,"18,264.47"


In [11]:
station_period = (
    sessions.groupby("station_id")
    .agg(
        first_session=("start_timestamp", "min"),
        last_session=("start_timestamp", "max")
    )
    .reset_index()
)

station_period["observed_days"] = (
    station_period["last_session"] -
    station_period["first_session"]
).dt.days + 1

station_performance = station_performance.merge(
    station_period,
    on="station_id",
    how="left"
)

station_performance["sessions_per_day"] = (
    station_performance["total_sessions"] /
    station_performance["observed_days"]
)

station_performance["energy_per_day"] = (
    station_performance["total_energy_kwh"] /
    station_performance["observed_days"]
)

station_performance["revenue_per_day"] = (
    station_performance["total_revenue"] /
    station_performance["observed_days"]
)

display(
    station_performance[
        [
            "station_id",
            "activation_year",
            "first_session",
            "last_session",
            "observed_days",
            "sessions_per_day",
            "energy_per_day",
            "revenue_per_day"
        ]
    ].head(10)
)

,station_id,activation_year,first_session,last_session,observed_days,sessions_per_day,energy_per_day,revenue_per_day
0,STN_001,2022,2022-01-01 01:30:00,2024-12-31 17:10:00,1096,13.83,566.56,328.39
1,STN_002,2022,2022-01-01 03:20:00,2024-12-31 20:20:00,1096,13.94,858.58,497.68
2,STN_003,2022,2022-01-01 07:30:00,2024-12-31 20:50:00,1096,13.98,669.53,387.41
3,STN_004,2022,2022-01-01 03:20:00,2024-12-31 20:40:00,1096,13.80,657.59,381.23
4,STN_005,2022,2022-01-01 07:50:00,2024-12-31 20:40:00,1096,13.83,640.56,370.67
5,STN_006,2022,2022-01-01 09:30:00,2024-12-31 20:30:00,1096,13.73,710.29,410.87
6,STN_007,2022,2022-01-01 02:50:00,2024-12-31 22:40:00,1096,13.89,607.95,352.14
7,STN_008,2022,2022-01-01 08:10:00,2024-12-31 18:50:00,1096,13.83,779.70,451.93
8,STN_009,2023,2023-01-01 00:40:00,2024-12-31 18:10:00,731,13.90,458.31,281.09
9,STN_010,2023,2023-01-01 08:20:00,2024-12-31 19:20:00,731,13.81,643.17,394.25


In [12]:
station_performance["sessions_per_charger_per_day"] = (
    station_performance["total_sessions"] /
    (station_performance["n_chargers"] *
     station_performance["observed_days"])
)

station_performance["energy_per_charger_per_day"] = (
    station_performance["total_energy_kwh"] /
    (station_performance["n_chargers"] *
     station_performance["observed_days"])
)

station_performance["revenue_per_charger_per_day"] = (
    station_performance["total_revenue"] /
    (station_performance["n_chargers"] *
     station_performance["observed_days"])
)

display(
    station_performance[
        [
            "station_id",
            "region",
            "activation_year",
            "n_chargers",
            "sessions_per_charger_per_day",
            "energy_per_charger_per_day",
            "revenue_per_charger_per_day"
        ]
    ].head(10)
)

,station_id,region,activation_year,n_chargers,sessions_per_charger_per_day,energy_per_charger_per_day,revenue_per_charger_per_day
0,STN_001,North West,2022,4,3.46,141.64,82.10
1,STN_002,London,2022,5,2.79,171.72,99.54
2,STN_003,Midlands,2022,5,2.80,133.91,77.48
3,STN_004,North West,2022,8,1.72,82.20,47.65
4,STN_005,North West,2022,8,1.73,80.07,46.33
5,STN_006,South West,2022,4,3.43,177.57,102.72
6,STN_007,North East,2022,4,3.47,151.99,88.03
7,STN_008,Midlands,2022,5,2.77,155.94,90.39
8,STN_009,Wales,2023,8,1.74,57.29,35.14
9,STN_010,North East,2023,8,1.73,80.40,49.28


In [13]:
adjusted_metrics = [
    "sessions_per_charger_per_day",
    "energy_per_charger_per_day",
    "revenue_per_charger_per_day"
]

for metric in adjusted_metrics:
    print(f"\nTop 5 stations by {metric}:")
    display(
        station_performance[
            ["station_id", "station_name", "region", metric]
        ]
        .sort_values(metric, ascending=False)
        .head(5)
    )

    print(f"Bottom 5 stations by {metric}:")
    display(
        station_performance[
            ["station_id", "station_name", "region", metric]
        ]
        .sort_values(metric)
        .head(5)
    )


Top 5 stations by sessions_per_charger_per_day:


,station_id,station_name,region,sessions_per_charger_per_day
29,STN_030,EV FastCharge 30,London,3.55
13,STN_014,EV FastCharge 14,London,3.48
6,STN_007,EV FastCharge 7,North East,3.47
21,STN_022,EV FastCharge 22,London,3.47
17,STN_018,EV FastCharge 18,South East,3.46


Bottom 5 stations by sessions_per_charger_per_day:


,station_id,station_name,region,sessions_per_charger_per_day
20,STN_021,EV FastCharge 21,South East,1.68
26,STN_027,EV FastCharge 27,South East,1.71
24,STN_025,EV FastCharge 25,North West,1.72
22,STN_023,EV FastCharge 23,Scotland,1.72
12,STN_013,EV FastCharge 13,Scotland,1.72



Top 5 stations by energy_per_charger_per_day:


,station_id,station_name,region,energy_per_charger_per_day
27,STN_028,EV FastCharge 28,Midlands,212.24
5,STN_006,EV FastCharge 6,South West,177.57
1,STN_002,EV FastCharge 2,London,171.72
17,STN_018,EV FastCharge 18,South East,165.27
13,STN_014,EV FastCharge 14,London,164.43


... (2 outputs left out)


Top 5 stations by revenue_per_charger_per_day:


,station_id,station_name,region,revenue_per_charger_per_day
27,STN_028,EV FastCharge 28,Midlands,124.64
5,STN_006,EV FastCharge 6,South West,102.72
13,STN_014,EV FastCharge 14,London,100.86
1,STN_002,EV FastCharge 2,London,99.54
11,STN_012,EV FastCharge 12,South West,97.05


Bottom 5 stations by revenue_per_charger_per_day:


,station_id,station_name,region,revenue_per_charger_per_day
8,STN_009,EV FastCharge 9,Wales,35.14
20,STN_021,EV FastCharge 21,South East,38.40
22,STN_023,EV FastCharge 23,Scotland,42.69
25,STN_026,EV FastCharge 26,London,43.72
4,STN_005,EV FastCharge 5,North West,46.33


In [14]:
score_metrics = [
    "sessions_per_charger_per_day",
    "energy_per_charger_per_day",
    "revenue_per_charger_per_day"
]

for metric in score_metrics:
    station_performance[f"{metric}_score"] = (
        station_performance[metric].rank(pct=True) * 100
    )

station_performance["performance_score"] = (
    station_performance[
        [f"{metric}_score" for metric in score_metrics]
    ].mean(axis=1)
)

station_performance["performance_rank"] = (
    station_performance["performance_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

station_ranking = station_performance.sort_values(
    "performance_rank"
)

display(
    station_ranking[
        [
            "station_id",
            "station_name",
            "region",
            "performance_score",
            "performance_rank"
        ]
    ]
)

,station_id,station_name,region,performance_score,performance_rank
13,STN_014,EV FastCharge 14,London,92.93,1
27,STN_028,EV FastCharge 28,Midlands,92.93,1
5,STN_006,EV FastCharge 6,South West,91.92,3
17,STN_018,EV FastCharge 18,South East,87.88,4
6,STN_007,EV FastCharge 7,North East,83.84,5
1,STN_002,EV FastCharge 2,London,81.82,6
11,STN_012,EV FastCharge 12,South West,79.80,7
30,STN_031,EV FastCharge 31,Midlands,75.76,8
7,STN_008,EV FastCharge 8,Midlands,73.74,9
0,STN_001,EV FastCharge 1,North West,73.74,9


In [15]:
score_columns = [
    "sessions_per_charger_per_day_score",
    "energy_per_charger_per_day_score",
    "revenue_per_charger_per_day_score"
]

display(
    pd.concat([
        station_ranking.head(5),
        station_ranking.tail(5)
    ])[
        [
            "station_id",
            "region",
            *score_columns,
            "performance_score",
            "performance_rank"
        ]
    ]
)

,station_id,region,sessions_per_charger_per_day_score,energy_per_charger_per_day_score,revenue_per_charger_per_day_score,performance_score,performance_rank
13,STN_014,London,96.97,87.88,93.94,92.93,1
27,STN_028,Midlands,78.79,100.00,100.00,92.93,1
5,STN_006,South West,81.82,96.97,96.97,91.92,3
17,STN_018,South East,87.88,90.91,84.85,87.88,4
6,STN_007,North East,93.94,78.79,78.79,83.84,5
12,STN_013,Scotland,15.15,21.21,27.27,21.21,29
25,STN_026,London,18.18,12.12,12.12,14.14,30
8,STN_009,Wales,33.33,3.03,3.03,13.13,31
22,STN_023,Scotland,12.12,9.09,9.09,10.10,32
20,STN_021,South East,3.03,6.06,6.06,5.05,33


In [16]:
scenarios = {
    "Balanced": [0.3333, 0.3333, 0.3334],
    "Demand_Focused": [0.50, 0.30, 0.20],
    "Revenue_Focused": [0.20, 0.30, 0.50]
}

score_columns = [
    "sessions_per_charger_per_day_score",
    "energy_per_charger_per_day_score",
    "revenue_per_charger_per_day_score"
]

sensitivity_results = station_performance[
    ["station_id", "station_name", "region"]
].copy()

for scenario, weights in scenarios.items():
    sensitivity_results[scenario] = (
        station_performance[score_columns]
        .mul(weights)
        .sum(axis=1)
    )

    sensitivity_results[f"{scenario}_rank"] = (
        sensitivity_results[scenario]
        .rank(ascending=False, method="min")
        .astype(int)
    )

display(
    sensitivity_results.sort_values("Balanced_rank")
)

,station_id,station_name,region,Balanced,Balanced_rank,Demand_Focused,Demand_Focused_rank,Revenue_Focused,Revenue_Focused_rank
27,STN_028,EV FastCharge 28,Midlands,92.93,1,89.39,2,95.76,1
13,STN_014,EV FastCharge 14,London,92.93,2,93.64,1,92.73,3
5,STN_006,EV FastCharge 6,South West,91.92,3,89.39,2,93.94,2
17,STN_018,EV FastCharge 18,South East,87.88,4,88.18,4,87.27,4
6,STN_007,EV FastCharge 7,North East,83.84,5,86.36,5,81.82,7
1,STN_002,EV FastCharge 2,London,81.82,6,76.67,7,85.76,5
11,STN_012,EV FastCharge 12,South West,79.80,7,76.36,9,82.73,6
30,STN_031,EV FastCharge 31,Midlands,75.76,8,75.76,10,75.76,9
7,STN_008,EV FastCharge 8,Midlands,73.74,9,69.70,14,76.97,8
0,STN_001,EV FastCharge 1,North West,73.74,10,76.67,8,71.21,11


In [17]:
activation_comparison = (
    station_performance.groupby("activation_year")
    .agg(
        total_stations=("station_id", "count"),
        average_chargers=("n_chargers", "mean"),
        average_sessions_per_charger_per_day=(
            "sessions_per_charger_per_day", "mean"
        ),
        average_energy_per_charger_per_day=(
            "energy_per_charger_per_day", "mean"
        ),
        average_revenue_per_charger_per_day=(
            "revenue_per_charger_per_day", "mean"
        ),
        average_performance_score=("performance_score", "mean")
    )
    .reset_index()
)

display(activation_comparison)

,activation_year,total_stations,average_chargers,average_sessions_per_charger_per_day,average_energy_per_charger_per_day,average_revenue_per_charger_per_day,average_performance_score
0,2022,8,5.38,2.77,136.88,79.28,64.27
1,2023,9,6.56,2.26,102.86,63.09,44.11
2,2024,16,6.06,2.48,112.47,66.15,49.31


In [18]:
regional_performance = (
    station_performance.groupby("region")
    .agg(
        total_stations=("station_id", "count"),
        average_sessions_per_charger_per_day=(
            "sessions_per_charger_per_day", "mean"
        ),
        average_energy_per_charger_per_day=(
            "energy_per_charger_per_day", "mean"
        ),
        average_revenue_per_charger_per_day=(
            "revenue_per_charger_per_day", "mean"
        ),
        average_performance_score=("performance_score", "mean")
    )
    .reset_index()
    .sort_values("average_performance_score", ascending=False)
)

display(regional_performance)

,region,total_stations,average_sessions_per_charger_per_day,average_energy_per_charger_per_day,average_revenue_per_charger_per_day,average_performance_score
6,South West,3,2.85,145.89,86.55,73.74
1,Midlands,5,2.82,145.88,85.67,65.66
0,London,5,3.00,128.37,76.08,65.45
2,North East,3,2.67,125.11,73.77,60.94
8,Yorkshire & Humber,2,2.40,107.25,65.75,47.98
5,South East,4,2.29,102.05,60.56,41.41
3,North West,7,2.13,97.33,56.87,39.39
4,Scotland,3,1.93,91.60,54.55,29.97
7,Wales,1,1.74,57.29,35.14,13.13


In [19]:
regional_consistency = (
    station_performance.groupby("region")
    .agg(
        total_stations=("station_id", "count"),
        minimum_score=("performance_score", "min"),
        maximum_score=("performance_score", "max"),
        average_score=("performance_score", "mean"),
        median_score=("performance_score", "median"),
        score_std=("performance_score", "std")
    )
    .reset_index()
    .sort_values("average_score", ascending=False)
)

display(regional_consistency)

,region,total_stations,minimum_score,maximum_score,average_score,median_score,score_std
6,South West,3,49.49,91.92,73.74,79.80,21.85
1,Midlands,5,22.22,92.93,65.66,73.74,26.47
0,London,5,14.14,92.93,65.45,71.72,30.40
2,North East,3,27.27,83.84,60.94,71.72,29.78
8,Yorkshire & Humber,2,26.26,69.70,47.98,47.98,30.71
5,South East,4,5.05,87.88,41.41,36.36,34.94
3,North West,7,22.22,73.74,39.39,36.36,18.23
4,Scotland,3,10.10,58.59,29.97,21.21,25.40
7,Wales,1,13.13,13.13,13.13,13.13,NaN


In [20]:
print(regional_consistency.columns.tolist())
display(regional_consistency.to_string(index=False))

['region', 'total_stations', 'minimum_score', 'maximum_score', 'average_score', 'median_score', 'score_std']


'            region  total_stations  minimum_score  maximum_score  average_score  median_score  score_std\n        South West               3          49.49          91.92          73.74         79.80      21.85\n          Midlands               5          22.22          92.93          65.66         73.74      26.47\n            London               5          14.14          92.93          65.45         71.72      30.40\n        North East               3          27.27          83.84          60.94         71.72      29.78\nYorkshire & Humber               2          26.26          69.70          47.98         47.98      30.71\n        South East               4           5.05          87.88          41.41         36.36      34.94\n        North West               7          22.22          73.74          39.39         36.36      18.23\n          Scotland               3          10.10          58.59          29.97         21.21      25.40\n             Wales               1          1

In [21]:
station_performance.columns.tolist()

['station_id',
 'station_name',
 'region',
 'activation_year',
 'n_chargers',
 'total_sessions',
 'total_energy_kwh',
 'total_revenue',
 'total_duration_minutes',
 'average_energy_per_session',
 'average_revenue_per_session',
 'sessions_per_charger',
 'energy_per_charger',
 'revenue_per_charger',
 'first_session',
 'last_session',
 'observed_days',
 'sessions_per_day',
 'energy_per_day',
 'revenue_per_day',
 'sessions_per_charger_per_day',
 'energy_per_charger_per_day',
 'revenue_per_charger_per_day',
 'sessions_per_charger_per_day_score',
 'energy_per_charger_per_day_score',
 'revenue_per_charger_per_day_score',
 'performance_score',
 'performance_rank']

In [22]:
final_station_performance = station_performance[
    [
        "station_id",
        "station_name",
        "region",
        "activation_year",
        "n_chargers",
        "total_sessions",
        "total_energy_kwh",
        "total_revenue",
        "total_duration_minutes",
        "average_energy_per_session",
        "average_revenue_per_session",
        "observed_days",
        "first_session",
        "last_session",
        "sessions_per_charger_per_day",
        "energy_per_charger_per_day",
        "revenue_per_charger_per_day",
        "sessions_per_charger_per_day_score",
        "energy_per_charger_per_day_score",
        "revenue_per_charger_per_day_score",
        "performance_score",
        "performance_rank"
    ]
].copy()

final_station_performance = final_station_performance.sort_values(
    "performance_rank"
).reset_index(drop=True)

display(final_station_performance.head(10))

,station_id,station_name,region,activation_year,n_chargers,total_sessions,total_energy_kwh,total_revenue,total_duration_minutes,average_energy_per_session,average_revenue_per_session,observed_days,first_session,last_session,sessions_per_charger_per_day,energy_per_charger_per_day,revenue_per_charger_per_day,sessions_per_charger_per_day_score,energy_per_charger_per_day_score,revenue_per_charger_per_day_score,performance_score,performance_rank
0,STN_014,EV FastCharge 14,London,2023,4,10161,"480,807.60","294,920.09",365259,47.32,29.02,731,2023-01-01 01:50:00,2024-12-31 21:30:00,3.48,164.43,100.86,96.97,87.88,93.94,92.93,1
1,STN_028,EV FastCharge 28,Midlands,2024,4,5000,"310,719.00","182,471.88",110097,62.14,36.49,366,2024-01-01 07:10:00,2024-12-31 19:20:00,3.42,212.24,124.64,78.79,100.00,100.00,92.93,1
2,STN_006,EV FastCharge 6,South West,2022,4,15045,"778,478.10","450,318.99",441068,51.74,29.93,1096,2022-01-01 09:30:00,2024-12-31 20:30:00,3.43,177.57,102.72,81.82,96.97,96.97,91.92,3
3,STN_018,EV FastCharge 18,South East,2024,4,5064,"241,949.70","142,055.25",181370,47.78,28.05,366,2024-01-01 04:20:00,2024-12-31 16:50:00,3.46,165.27,97.03,87.88,90.91,84.85,87.88,4
4,STN_007,EV FastCharge 7,North East,2022,4,15218,"666,315.70","385,943.19",636681,43.78,25.36,1096,2022-01-01 02:50:00,2024-12-31 22:40:00,3.47,151.99,88.03,93.94,78.79,78.79,83.84,5
5,STN_002,EV FastCharge 2,London,2022,5,15276,"940,999.00","545,458.72",405401,61.60,35.71,1096,2022-01-01 03:20:00,2024-12-31 20:20:00,2.79,171.72,99.54,60.61,93.94,90.91,81.82,6
6,STN_012,EV FastCharge 12,South West,2023,5,10231,"578,262.80","354,722.41",280642,56.52,34.67,731,2023-01-01 03:40:00,2024-12-31 21:40:00,2.80,158.21,97.05,66.67,84.85,87.88,79.80,7
7,STN_031,EV FastCharge 31,Midlands,2024,4,4999,"217,616.20","128,238.83",210103,43.53,25.65,366,2024-01-01 06:20:00,2024-12-31 23:10:00,3.41,148.64,87.59,75.76,75.76,75.76,75.76,8
8,STN_008,EV FastCharge 8,Midlands,2022,5,15154,"854,549.00","495,315.18",415722,56.39,32.69,1096,2022-01-01 08:10:00,2024-12-31 18:50:00,2.77,155.94,90.39,57.58,81.82,81.82,73.74,9
9,STN_001,EV FastCharge 1,North West,2022,4,15162,"620,951.60","359,914.16",560630,40.95,23.74,1096,2022-01-01 01:30:00,2024-12-31 17:10:00,3.46,141.64,82.10,84.85,69.70,66.67,73.74,9


In [23]:
print("Shape:", final_station_performance.shape)
print("Unique stations:", final_station_performance["station_id"].nunique())
print("Missing values:", final_station_performance.isnull().sum().sum())
print("Duplicate station IDs:", final_station_performance["station_id"].duplicated().sum())

print(
    "Total sessions:",
    final_station_performance["total_sessions"].sum()
)

print(
    "Total energy:",
    final_station_performance["total_energy_kwh"].sum()
)

print(
    "Total revenue:",
    final_station_performance["total_revenue"].sum()
)

Shape: (33, 22)
Unique stations: 33
Missing values: 0
Duplicate station IDs: 0
Total sessions: 294024
Total energy: 13808745.799999999
Total revenue: 8171100.12


In [24]:
from pathlib import Path

output_dir = Path("analysis")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "station_performance_benchmark.csv"

final_station_performance.to_csv(output_path, index=False)

print(f"Dataset exported successfully: {output_path}")

Dataset exported successfully: analysis\station_performance_benchmark.csv


In [25]:
verification_df = pd.read_csv(output_path)

print("Exported shape:", verification_df.shape)
print("Unique stations:", verification_df["station_id"].nunique())
print("Missing values:", verification_df.isnull().sum().sum())

display(verification_df.head())

Exported shape: (33, 22)
Unique stations: 33
Missing values: 0


,station_id,station_name,region,activation_year,n_chargers,total_sessions,total_energy_kwh,total_revenue,total_duration_minutes,average_energy_per_session,average_revenue_per_session,observed_days,first_session,last_session,sessions_per_charger_per_day,energy_per_charger_per_day,revenue_per_charger_per_day,sessions_per_charger_per_day_score,energy_per_charger_per_day_score,revenue_per_charger_per_day_score,performance_score,performance_rank
0,STN_014,EV FastCharge 14,London,2023,4,10161,"480,807.60","294,920.09",365259,47.32,29.02,731,2023-01-01 01:50:00,2024-12-31 21:30:00,3.48,164.43,100.86,96.97,87.88,93.94,92.93,1
1,STN_028,EV FastCharge 28,Midlands,2024,4,5000,"310,719.00","182,471.88",110097,62.14,36.49,366,2024-01-01 07:10:00,2024-12-31 19:20:00,3.42,212.24,124.64,78.79,100.00,100.00,92.93,1
2,STN_006,EV FastCharge 6,South West,2022,4,15045,"778,478.10","450,318.99",441068,51.74,29.93,1096,2022-01-01 09:30:00,2024-12-31 20:30:00,3.43,177.57,102.72,81.82,96.97,96.97,91.92,3
3,STN_018,EV FastCharge 18,South East,2024,4,5064,"241,949.70","142,055.25",181370,47.78,28.05,366,2024-01-01 04:20:00,2024-12-31 16:50:00,3.46,165.27,97.03,87.88,90.91,84.85,87.88,4
4,STN_007,EV FastCharge 7,North East,2022,4,15218,"666,315.70","385,943.19",636681,43.78,25.36,1096,2022-01-01 02:50:00,2024-12-31 22:40:00,3.47,151.99,88.03,93.94,78.79,78.79,83.84,5
